In [ ]:
import pandas as pd
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')

print(f'Train: {train.shape}')
print(f'Valid: {valid.shape}')
print(f'Test:  {test.shape}')

Train: (5082, 11)
Valid: (565, 11)
Test:  (997, 7)


In [ ]:
import torch
import numpy as np
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Χρησιμοποιούμε: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Μνήμη GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Χρησιμοποιούμε: cuda
GPU: Tesla T4
Μνήμη GPU: 15.6 GB


In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product,
                       verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard,
                         average='macro', zero_division=0)
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)

    correct_hazard_mask = (y_true_hazard == y_pred_hazard)
    n_correct = correct_hazard_mask.sum()

    if n_correct == 0:
        f1_product = 0.0
    else:
        f1_product = f1_score(
            y_true_product[correct_hazard_mask],
            y_pred_product[correct_hazard_mask],
            average='macro', zero_division=0
        )
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {n_correct}/{len(y_true_hazard)} ({100*n_correct/len(y_true_hazard):.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [ ]:
# Texts
texts_train = (train['title'].fillna('') + ' ' + train['text'].fillna('').str[:550]).tolist()
texts_valid = (valid['title'].fillna('') + ' ' + valid['text'].fillna('').str[:550]).tolist()
texts_test  = (test['title'].fillna('')  + ' ' + test['text'].fillna('').str[:550]).tolist()

texts_train = [str(t) for t in texts_train]
texts_valid = [str(t) for t in texts_valid]
texts_test  = [str(t) for t in texts_test]

# Label Encoding
le_hazard  = LabelEncoder()
le_product = LabelEncoder()

y_train_hazard_enc  = le_hazard.fit_transform(train['hazard-category'])
y_valid_hazard_enc  = le_hazard.transform(valid['hazard-category'])
y_train_product_enc = le_product.fit_transform(train['product-category'])
y_valid_product_enc = le_product.transform(valid['product-category'])

print(f'Hazard classes:  {len(le_hazard.classes_)}')
print(f'Product classes: {len(le_product.classes_)}')

Hazard classes:  10
Product classes: 22


In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(label, dtype=torch.long)
        }

print('Dataset class έτοιμο')

Dataset class έτοιμο


In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded


In [ ]:
# Hazard datasets
train_dataset_hazard = FoodHazardDataset(texts_train, y_train_hazard_enc, tokenizer)
valid_dataset_hazard = FoodHazardDataset(texts_valid, y_valid_hazard_enc, tokenizer)

# Product datasets
train_dataset_product = FoodHazardDataset(texts_train, y_train_product_enc, tokenizer)
valid_dataset_product = FoodHazardDataset(texts_valid, y_valid_product_enc, tokenizer)

# DataLoaders — batch_size=32 γιατί έχουμε GPU με 15.6GB!
train_loader_hazard   = DataLoader(train_dataset_hazard,   batch_size=32, shuffle=True)
valid_loader_hazard   = DataLoader(valid_dataset_hazard,   batch_size=32, shuffle=False)
train_loader_product  = DataLoader(train_dataset_product,  batch_size=32, shuffle=True)
valid_loader_product  = DataLoader(valid_dataset_product,  batch_size=32, shuffle=False)

print(f'Train batches (hazard):  {len(train_loader_hazard)}')
print(f'Valid batches (hazard):  {len(valid_loader_hazard)}')

Train batches (hazard):  159
Valid batches (hazard):  18


In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 30 == 0:
            print(f'  Batch {batch_idx}/{len(loader)} — Loss: {loss.item():.4f}', end='\r')

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    return np.array(all_preds)

In [ ]:
# Φόρτωση μοντέλου για hazard
bert_hazard = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le_hazard.classes_)
).to(device)

optimizer_hazard = AdamW(bert_hazard.parameters(), lr=2e-5)

print('=== FINE-TUNING DISTILBERT — HAZARD ===\n')
EPOCHS = 3

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    train_loss = train_epoch(bert_hazard, train_loader_hazard, optimizer_hazard)

    preds_enc = evaluate(bert_hazard, valid_loader_hazard)
    preds     = le_hazard.inverse_transform(preds_enc)
    f1        = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Hazard: {f1:.4f}')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== FINE-TUNING DISTILBERT — HAZARD ===

Epoch 1/3

  Loss: 0.9872 | macro-F1 Hazard: 0.4477
Epoch 2/3

  Loss: 0.3651 | macro-F1 Hazard: 0.5614
Epoch 3/3

  Loss: 0.2361 | macro-F1 Hazard: 0.6885


In [ ]:
print('=== ΣΥΝΕΧΕΙΑ FINE-TUNING — EPOCHS 4-5 ===\n')
EXTRA_EPOCHS = 2

for epoch in range(EXTRA_EPOCHS):
    print(f'Epoch {epoch+4}/{5}')
    train_loss = train_epoch(bert_hazard, train_loader_hazard, optimizer_hazard)

    preds_enc = evaluate(bert_hazard, valid_loader_hazard)
    preds     = le_hazard.inverse_transform(preds_enc)
    f1        = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Hazard: {f1:.4f}')

=== ΣΥΝΕΧΕΙΑ FINE-TUNING — EPOCHS 4-5 ===

Epoch 4/5

  Loss: 0.0957 | macro-F1 Hazard: 0.7237
Epoch 5/5

  Loss: 0.0691 | macro-F1 Hazard: 0.8042


In [ ]:
print('=== ΣΥΝΕΧΕΙΑ FINE-TUNING — EPOCHS 6-7 ===\n')
EXTRA_EPOCHS = 2

for epoch in range(EXTRA_EPOCHS):
    print(f'Epoch {epoch+6}/{7}')
    train_loss = train_epoch(bert_hazard, train_loader_hazard, optimizer_hazard)

    preds_enc = evaluate(bert_hazard, valid_loader_hazard)
    preds     = le_hazard.inverse_transform(preds_enc)
    f1        = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Hazard: {f1:.4f}')

=== ΣΥΝΕΧΕΙΑ FINE-TUNING — EPOCHS 6-7 ===

Epoch 6/7

  Loss: 0.0505 | macro-F1 Hazard: 0.8054
Epoch 7/7

  Loss: 0.0391 | macro-F1 Hazard: 0.8552


In [ ]:
print('=== ΣΥΝΕΧΕΙΑ FINE-TUNING — EPOCHS 8-9 ===\n')
EXTRA_EPOCHS = 2

for epoch in range(EXTRA_EPOCHS):
    print(f'Epoch {epoch+8}/{9}')
    train_loss = train_epoch(bert_hazard, train_loader_hazard, optimizer_hazard)

    preds_enc = evaluate(bert_hazard, valid_loader_hazard)
    preds     = le_hazard.inverse_transform(preds_enc)
    f1        = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Hazard: {f1:.4f}')

=== ΣΥΝΕΧΕΙΑ FINE-TUNING — EPOCHS 8-9 ===

Epoch 8/9

  Loss: 0.0310 | macro-F1 Hazard: 0.8213
Epoch 9/9

  Loss: 0.0264 | macro-F1 Hazard: 0.8213


In [ ]:
# Επανφόρτωση μοντέλου από την αρχή
bert_hazard = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le_hazard.classes_)
).to(device)

optimizer_hazard = AdamW(bert_hazard.parameters(), lr=2e-5)

print('=== FINE-TUNING ΜΕ EARLY STOPPING — HAZARD ===\n')

best_f1    = 0
best_epoch = 0
EPOCHS     = 9  # ξέρουμε ότι το βέλτιστο είναι γύρω στο epoch 7

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    train_loss = train_epoch(bert_hazard, train_loader_hazard, optimizer_hazard)

    preds_enc = evaluate(bert_hazard, valid_loader_hazard)
    preds     = le_hazard.inverse_transform(preds_enc)
    f1        = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Hazard: {f1:.4f}')

    # Αποθήκευση καλύτερου μοντέλου
    if f1 > best_f1:
        best_f1    = f1
        best_epoch = epoch + 1
        torch.save(bert_hazard.state_dict(), 'best_hazard_model.pt')
        print(f'   Νέο καλύτερο! Αποθηκεύτηκε (epoch {best_epoch}, F1={best_f1:.4f})')
    else:
        print(f'  Δεν βελτιώθηκε (καλύτερο: epoch {best_epoch}, F1={best_f1:.4f})')

# Φόρτωση καλύτερου μοντέλου
bert_hazard.load_state_dict(torch.load('best_hazard_model.pt'))
print(f'\nΤελικό μοντέλο: Epoch {best_epoch} με macro-F1={best_f1:.4f}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== FINE-TUNING ΜΕ EARLY STOPPING — HAZARD ===

Epoch 1/9

  Loss: 1.0153 | macro-F1 Hazard: 0.4499
  ✅ Νέο καλύτερο! Αποθηκεύτηκε (epoch 1, F1=0.4499)
Epoch 2/9

  Loss: 0.3536 | macro-F1 Hazard: 0.6509
  ✅ Νέο καλύτερο! Αποθηκεύτηκε (epoch 2, F1=0.6509)
Epoch 3/9

  Loss: 0.2271 | macro-F1 Hazard: 0.7100
  ✅ Νέο καλύτερο! Αποθηκεύτηκε (epoch 3, F1=0.7100)
Epoch 4/9

  Loss: 0.1692 | macro-F1 Hazard: 0.7012
  ⚠️ Δεν βελτιώθηκε (καλύτερο: epoch 3, F1=0.7100)
Epoch 5/9

  Loss: 0.1283 | macro-F1 Hazard: 0.6869
  ⚠️ Δεν βελτιώθηκε (καλύτερο: epoch 3, F1=0.7100)
Epoch 6/9

  Loss: 0.0963 | macro-F1 Hazard: 0.7035
  ⚠️ Δεν βελτιώθηκε (καλύτερο: epoch 3, F1=0.7100)
Epoch 7/9

  Loss: 0.0715 | macro-F1 Hazard: 0.8142
  ✅ Νέο καλύτερο! Αποθηκεύτηκε (epoch 7, F1=0.8142)
Epoch 8/9

  Loss: 0.0620 | macro-F1 Hazard: 0.8348
  ✅ Νέο καλύτερο! Αποθηκεύτηκε (epoch 8, F1=0.8348)
Epoch 9/9

  Loss: 0.0447 | macro-F1 Hazard: 0.8556
  ✅ Νέο καλύτερο! Αποθηκεύτηκε (epoch 9, F1=0.8556)

Τελικό μοντέλο: Ep

In [ ]:
print('=== ΣΥΝΕΧΕΙΑ — EPOCHS 10-11 ===\n')

for epoch in range(2):
    print(f'Epoch {epoch+10}/11')
    train_loss = train_epoch(bert_hazard, train_loader_hazard, optimizer_hazard)

    preds_enc = evaluate(bert_hazard, valid_loader_hazard)
    preds     = le_hazard.inverse_transform(preds_enc)
    f1        = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Hazard: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(bert_hazard.state_dict(), 'best_hazard_model.pt')
        print(f' Νέο καλύτερο, Αποθηκεύτηκε (F1={best_f1:.4f})')
    else:
        print(f' Δεν βελτιώθηκε (καλύτερο: F1={best_f1:.4f})')

=== ΣΥΝΕΧΕΙΑ — EPOCHS 10-11 ===

Epoch 10/11

  Loss: 0.0395 | macro-F1 Hazard: 0.8467
 Δεν βελτιώθηκε (καλύτερο: F1=0.8556)
Epoch 11/11

  Loss: 0.0332 | macro-F1 Hazard: 0.8138
 Δεν βελτιώθηκε (καλύτερο: F1=0.8556)


In [ ]:
# Φόρτωση νέου μοντέλου για product
bert_product = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le_product.classes_)
).to(device)

optimizer_product = AdamW(bert_product.parameters(), lr=2e-5)

print('=== FINE-TUNING ΜΕ EARLY STOPPING — PRODUCT ===\n')

best_f1_product    = 0
best_epoch_product = 0
EPOCHS             = 9  # ίδιος αριθμός με hazard

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    train_loss = train_epoch(bert_product, train_loader_product, optimizer_product)

    preds_enc = evaluate(bert_product, valid_loader_product)
    preds     = le_product.inverse_transform(preds_enc)
    f1        = f1_score(valid['product-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Product: {f1:.4f}')

    if f1 > best_f1_product:
        best_f1_product    = f1
        best_epoch_product = epoch + 1
        torch.save(bert_product.state_dict(), 'best_product_model.pt')
        print(f'  Νέο καλύτερο, Αποθηκεύτηκε (epoch {best_epoch_product}, F1={best_f1_product:.4f})')
    else:
        print(f'  Δεν βελτιώθηκε (καλύτερο: epoch {best_epoch_product}, F1={best_f1_product:.4f})')

# Φόρτωση καλύτερου μοντέλου
bert_product.load_state_dict(torch.load('best_product_model.pt'))
print(f'\nΤελικό μοντέλο: Epoch {best_epoch_product} με macro-F1={best_f1_product:.4f}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== FINE-TUNING ΜΕ EARLY STOPPING — PRODUCT ===

Epoch 1/9

  Loss: 2.0756 | macro-F1 Product: 0.3639
  Νέο καλύτερο, Αποθηκεύτηκε (epoch 1, F1=0.3639)
Epoch 2/9

  Loss: 1.0887 | macro-F1 Product: 0.5463
  Νέο καλύτερο, Αποθηκεύτηκε (epoch 2, F1=0.5463)
Epoch 3/9

  Loss: 0.7246 | macro-F1 Product: 0.6039
  Νέο καλύτερο, Αποθηκεύτηκε (epoch 3, F1=0.6039)
Epoch 4/9

  Loss: 0.5336 | macro-F1 Product: 0.6179
  Νέο καλύτερο, Αποθηκεύτηκε (epoch 4, F1=0.6179)
Epoch 5/9

  Loss: 0.3888 | macro-F1 Product: 0.6570
  Νέο καλύτερο, Αποθηκεύτηκε (epoch 5, F1=0.6570)
Epoch 6/9

  Loss: 0.2963 | macro-F1 Product: 0.6544
  Δεν βελτιώθηκε (καλύτερο: epoch 5, F1=0.6570)
Epoch 7/9

  Loss: 0.2161 | macro-F1 Product: 0.6470
  Δεν βελτιώθηκε (καλύτερο: epoch 5, F1=0.6570)
Epoch 8/9

  Loss: 0.1440 | macro-F1 Product: 0.6580
  Νέο καλύτερο, Αποθηκεύτηκε (epoch 8, F1=0.6580)
Epoch 9/9

  Loss: 0.1061 | macro-F1 Product: 0.6976
  Νέο καλύτερο, Αποθηκεύτηκε (epoch 9, F1=0.6976)

Τελικό μοντέλο: Epoch 9 με 

In [ ]:
print('=== ΣΥΝΕΧΕΙΑ PRODUCT — EPOCHS 10-11 ===\n')

for epoch in range(2):
    print(f'Epoch {epoch+10}/11')
    train_loss = train_epoch(bert_product, train_loader_product, optimizer_product)

    preds_enc = evaluate(bert_product, valid_loader_product)
    preds     = le_product.inverse_transform(preds_enc)
    f1        = f1_score(valid['product-category'], preds, average='macro', zero_division=0)

    print(f'\n  Loss: {train_loss:.4f} | macro-F1 Product: {f1:.4f}')

    if f1 > best_f1_product:
        best_f1_product = f1
        torch.save(bert_product.state_dict(), 'best_product_model.pt')
        print(f'  Νέο καλύτερο, Αποθηκεύτηκε (F1={best_f1_product:.4f})')
    else:
        print(f'  Δεν βελτιώθηκε (καλύτερο: F1={best_f1_product:.4f})')

=== ΣΥΝΕΧΕΙΑ PRODUCT — EPOCHS 10-11 ===

Epoch 10/11

  Loss: 0.0822 | macro-F1 Product: 0.6775
  Δεν βελτιώθηκε (καλύτερο: F1=0.6976)
Epoch 11/11

  Loss: 0.0645 | macro-F1 Product: 0.6760
  Δεν βελτιώθηκε (καλύτερο: F1=0.6976)


In [ ]:
# Φόρτωση καλύτερων μοντέλων
bert_hazard.load_state_dict(torch.load('best_hazard_model.pt'))
bert_product.load_state_dict(torch.load('best_product_model.pt'))

# Predictions στο validation set
preds_hazard_enc  = evaluate(bert_hazard, valid_loader_hazard)
preds_product_enc = evaluate(bert_product, valid_loader_product)

preds_hazard_bert  = le_hazard.inverse_transform(preds_hazard_enc)
preds_product_bert = le_product.inverse_transform(preds_product_enc)

print('=== ΑΞΙΟΛΟΓΗΣΗ FINE-TUNED DISTILBERT ===\n')
score_bert_ft = official_st1_score(
    valid['hazard-category'], preds_hazard_bert,
    valid['product-category'], preds_product_bert
)

print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'TF-IDF + SVM (tuned):        0.7436')
print(f'Fine-tuned DistilBERT:       {score_bert_ft:.4f}')

=== ΑΞΙΟΛΟΓΗΣΗ FINE-TUNED DISTILBERT ===

macro-F1 Hazard:                    0.8556
Σωστά hazard predictions:           523/565 (92.6%)
macro-F1 Product (given correct h): 0.7439
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.7998

=== ΣΥΓΚΡΙΣΗ ===
TF-IDF + SVM (tuned):        0.7436
Fine-tuned DistilBERT:       0.7998


In [ ]:
# Δημιουργία test dataset
from torch.utils.data import DataLoader

# Dummy labels για το test (δεν έχει labels)
dummy_labels = np.zeros(len(texts_test), dtype=int)

test_dataset_hazard  = FoodHazardDataset(texts_test, dummy_labels, tokenizer)
test_dataset_product = FoodHazardDataset(texts_test, dummy_labels, tokenizer)

test_loader_hazard  = DataLoader(test_dataset_hazard,  batch_size=32, shuffle=False)
test_loader_product = DataLoader(test_dataset_product, batch_size=32, shuffle=False)

# Predictions στο test set
preds_test_hazard_enc  = evaluate(bert_hazard,  test_loader_hazard)
preds_test_product_enc = evaluate(bert_product, test_loader_product)

preds_test_hazard  = le_hazard.inverse_transform(preds_test_hazard_enc)
preds_test_product = le_product.inverse_transform(preds_test_product_enc)

# Δημιουργία submission
submission_bert = pd.DataFrame({
    'id': test['id'],
    'hazard-category': preds_test_hazard,
    'product-category': preds_test_product
})

submission_bert.to_csv('submission_bert_finetuned.csv', index=False)
print('Submission file saved: submission_bert_finetuned.csv')
print(f'Αριθμός predictions: {len(submission_bert)}')
print('\n--- Πρώτες 5 προβλέψεις ---')
print(submission_bert.head())

Submission file saved: submission_bert_finetuned.csv
Αριθμός predictions: 997

--- Πρώτες 5 προβλέψεις ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological    prepared dishes and snacks
3   3      biological  meat, egg and dairy products
4   4  foreign bodies             ices and desserts


## Fine-tuning DistilBERT (Google Colab + GPU)

Επανεκπαίδευση DistilBERT με fine-tuning στο Google Colab
χρησιμοποιώντας Tesla T4 GPU (15.6GB).

**Εκπαίδευση:**
- Optimizer: AdamW, lr=2e-5
- Batch size: 32 (λόγω GPU)
- Early stopping: αποθήκευση καλύτερου epoch

**Αποτελέσματα:**

| Epoch | macro-F1 Hazard | macro-F1 Product |
|---|---|---|
| 1 | 0.4499 | 0.3639 |
| 3 | 0.7100 | 0.6039 |
| 5 | 0.8042 | 0.6570 |
| 7 | 0.8552 | 0.6470 |
| **9** | **0.8556**  | **0.6976**  |

**Validation Score: 0.7998**
**Kaggle Score: 0.72357**

**Παρατήρηση — Overfitting:**
Μεγάλη διαφορά validation (0.7998) vs Kaggle (0.72357).
Το μοντέλο εκπαιδεύτηκε για πολλά epochs με early stopping
βασισμένο στο validation set — αυτό προκάλεσε διαρροή
πληροφορίας από το validation στο μοντέλο.
Το TF-IDF+SVM (0.74078) παραμένει καλύτερο στο Kaggle.

In [ ]:
# Αποθήκευση BERT predictions για ensemble
# Validation set
bert_valid_hazard  = preds_hazard_bert
bert_valid_product = preds_product_bert

# Test set
bert_test_hazard  = preds_test_hazard
bert_test_product = preds_test_product

# Αποθήκευση σε CSV
bert_valid_preds = pd.DataFrame({
    'hazard_bert': bert_valid_hazard,
    'product_bert': bert_valid_product
})

bert_test_preds = pd.DataFrame({
    'id': test['id'],
    'hazard_bert': bert_test_hazard,
    'product_bert': bert_test_product
})

bert_valid_preds.to_csv('bert_valid_predictions.csv', index=False)
bert_test_preds.to_csv('bert_test_predictions.csv', index=False)

print('   bert_valid_predictions.csv')
print('   bert_test_predictions.csv')
print(f'\nValidation predictions: {len(bert_valid_preds)}')
print(f'Test predictions:       {len(bert_test_preds)}')

   bert_valid_predictions.csv
   bert_test_predictions.csv

Validation predictions: 565
Test predictions:       997


In [ ]:
# Εξαγωγή BERT probabilities
def get_bert_probs(model, loader):
    """
    Επιστρέφει τις πιθανότητες για κάθε κλάση (softmax).
    """
    model.eval()
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # Softmax για να πάρουμε πιθανότητες
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            all_probs.append(probs)

    return np.vstack(all_probs)

# Validation probabilities
bert_valid_hazard_probs  = get_bert_probs(bert_hazard, valid_loader_hazard)
bert_valid_product_probs = get_bert_probs(bert_product, valid_loader_product)

# Test probabilities
bert_test_hazard_probs  = get_bert_probs(bert_hazard, test_loader_hazard)
bert_test_product_probs = get_bert_probs(bert_product, test_loader_product)

print(f'BERT valid hazard probs shape:  {bert_valid_hazard_probs.shape}')
print(f'BERT valid product probs shape: {bert_valid_product_probs.shape}')

# Αποθήκευση
np.save('bert_valid_hazard_probs.npy', bert_valid_hazard_probs)
np.save('bert_valid_product_probs.npy', bert_valid_product_probs)
np.save('bert_test_hazard_probs.npy', bert_test_hazard_probs)
np.save('bert_test_product_probs.npy', bert_test_product_probs)

BERT valid hazard probs shape:  (565, 10)
BERT valid product probs shape: (565, 22)
